# Lab 5 Working with Data on the Command Line

Lab 5's matrix class is the main assignment. This shorter piece is what we work through in
the lab session, and it is a different skill: finding out what is in a data file, and
getting it into shape, before any Python is involved.

Lecture 11 starts from a real grades file that is a mess — a line of notes where the header
should be, words like `no solution` in columns that should hold numbers, scores written as
`68-sent` — and cleans it with pandas. The same jobs, and almost always the *first* look at
a new file, can be done faster with the Unix tools already on your machine. They also work
when the file is far too big to open.

Lab 3 covered moving around the filesystem: `ls`, `cd`, `mkdir`, `cp`, `mv`. Here we chain
commands together to ask questions of a file and to write a cleaned copy of it.

## The data

Nothing to download. Everything is in the repository you pulled:

| File | What it is |
|---|---|
| `Labs/Lab.5/bank-churn.csv` | 20,000 bank customers: name, country, age, balance, and whether they left. Mixed text and numbers. |
| `Labs/Lab.5/heart.csv` | 303 patients, 14 numeric columns, and a 0/1 diagnosis. |
| `Labs/Lab.5/2012_Workplace_Fatalities_by_State.csv` | Workplace fatalities per US state. Awkward on purpose — see the last exercise. |
| `Lectures/Lecture.11/Data-1401-Grades.csv` | The messy grades file from Lecture 11. |

The bank file is a 20,000-row extract of a Kaggle playground dataset (Binary Classification
with a Bank Churn Dataset, 2024); the heart and fatalities files are included with Lab 5.

## Finding the commands

`man <command>` is the manual for any of these: space scrolls, `q` quits. Some resources
found by searching:

* Paths and wildcards: https://www.warp.dev/terminus/linux-wildcards
* Introduction to the shell: https://github-pages.ucl.ac.uk/RCPSTrainingMaterials/HPCandHTCusingLegion/2_intro_to_shell.html
* Chaining commands: https://www.geeksforgeeks.org/chaining-commands-in-linux/
* Piping: https://www.geeksforgeeks.org/piping-in-unix-or-linux/
* Using `sed`: https://www.geeksforgeeks.org/sed-command-linux-set-2/
* Cheat sheet: https://cheatography.com/davechild/cheat-sheets/linux-command-line/

## How to work

Use a real command prompt: the WSL Ubuntu terminal on Windows, Terminal on macOS. When a
command does what you want, paste it into the cell under the exercise. A cell starting with
`!` runs a shell command from inside Jupyter — that is how Lecture 11 ran `cat` and `wc` —
so your pasted commands will run here too.

Three things to know first:

* `>` sends output into a file, replacing what was there.
* `>>` appends to the end of a file instead.
* `|` (a pipe) feeds one command's output straight into the next.


## Exercise 0: Somewhere to work

Make a directory for this lab inside your fork and copy the four CSV files into it. Work on
the copies, so a mistake never touches the originals and `git status` stays clean.

Hints: `mkdir`, then `cp`. A wildcard saves typing, and `cp` takes several files at once if
the last argument is a directory.

In [1]:
%%bash

# Created a seperate directory for the coommand line exercises then copied the 4 datasets
# into this folder using cp, so I can work with files and not change the original
# ls was used to confirm that all four files were copied succesfully.


mkdir -p commandline_work

cp bank-churn.csv commandline_work/
cp heart.csv commandline_work/
cp 2012_Workplace_Fatalities_by_State.csv commandline_work/
cp ../../Lectures/Lecture.11/Data-1401-Grades.csv commandline_work/

ls -lh commandline_work

total 2960
-rw-r--r--  1 joshuasthomas  staff   4.8K Nov 25 17:01 2012_Workplace_Fatalities_by_State.csv
-rw-r--r--  1 joshuasthomas  staff   1.4M Nov 25 17:01 bank-churn.csv
-rw-r--r--  1 joshuasthomas  staff   8.7K Nov 25 17:01 Data-1401-Grades.csv
-rw-r--r--  1 joshuasthomas  staff    11K Nov 25 17:01 heart.csv


## Exercise 1: First look at a file you have never seen

You have `bank-churn.csv` and know nothing about it. Answer these at the prompt:

1. How big is the file, and how many rows does it have?
2. What are the column names, and how many columns are there?
3. What do the first few rows actually look like?

Hints: `du -h` for size, `wc -l` for lines. `head -1` gets the header; piping it into
`tr ',' '\n'` puts each column name on its own line, and piping *that* into `nl` numbers
them — which you will want in the later exercises, where `cut` asks for column numbers.

In [2]:

%%bash

# I used cd to enter the folder containing my copied datasets. then du to check the file size and wc to count the lines.
# I displayed the column names and numbered them using nl and used head to examine the first few rows of the dataset.

cd commandline_work

echo "File size:"
du -h bank-churn.csv

echo "Number of lines:"
wc -l bank-churn.csv

echo "Column names:"
head -1 bank-churn.csv | tr ',' '\n' | nl

echo "Number of columns:"
head -1 bank-churn.csv | tr ',' '\n' | wc -l

echo "First 5 data records:"
head -6 bank-churn.csv


File size:
1.4M	bank-churn.csv
Number of lines:
   20001 bank-churn.csv
Column names:
     1	id
     2	CustomerId
     3	Surname
     4	CreditScore
     5	Geography
     6	Gender
     7	Age
     8	Tenure
     9	Balance
    10	NumOfProducts
    11	HasCrCard
    12	IsActiveMember
    13	EstimatedSalary
    14	Exited
Number of columns:
      14
First 5 data records:
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.0,2,1.0,0.0,181449.97,0
1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.0,2,1.0,1.0,49503.5,0
2,15694510,Hsueh,678,France,Male,40.0,10,0.0,2,1.0,0.0,184866.69,0
3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.0,2,1.0,1.0,15068.83,0


In [ ]:
# I examined the bank cxhurn dataset with command-line commands, the file has 20k customer records, not including the header. 
# Using du to determine file size, wc to count the lines, and (head,tr,nl) to find number of columns and 
#then displayed first 5 records to see how the data was organized.

## Exercise 2: Split a big file into three

Files too big to open are routine, and a common first move is to cut one into pieces.

Make 3 new CSV files, each with about a third of `bank-churn.csv`'s rows, and **each
starting with the header line**, so every piece is a readable CSV on its own.

Hints:

* `head -1 ... > part1.csv` starts a file with just the header. Do that three times.
* `wc -l` tells you how many rows there are to divide up. Remember one of them is the header.
* `head -N file | tail -M` selects a range: M lines ending at line N.
* Append each range to the right file with `>>`.
* Check your work: the three files' rows should add back up, `wc -l` on all three at once.

In [3]:

%%bash

# Exercise 2
# I divided the bank dataset into three smaller files.
# I used wc to count the rows and calculated how many records should be placed in each file.
# I used head and tail to select different sections and added the original header to each new CSV file.
# Finally, I checked the row counts to verify the split.

cd commandline_work

# Count customer records without the header
total=$(wc -l < bank-churn.csv)
rows=$((total - 1))

# Divide the records into three parts
part1=$(((rows + 2) / 3))
part2=$(((rows + 1) / 3))

# Create the three files with the original header
head -n 1 bank-churn.csv > part1.csv
head -n 1 bank-churn.csv > part2.csv
head -n 1 bank-churn.csv > part3.csv

# Add the customer records to each file
tail -n +2 bank-churn.csv | head -n "$part1" >> part1.csv

tail -n +$((part1 + 2)) bank-churn.csv |
    head -n "$part2" >> part2.csv

tail -n +$((part1 + part2 + 2)) bank-churn.csv >> part3.csv

# Check the number of lines in each file
echo "Original file and split file counts:"
wc -l bank-churn.csv part1.csv part2.csv part3.csv

# Check that all customer records were included
split_rows=$(($(wc -l < part1.csv) +
              $(wc -l < part2.csv) +
              $(wc -l < part3.csv) - 3))

echo "Original customer records: $rows"
echo "Customer records after splitting: $split_rows"

# Verify that the three files contain all records
test "$rows" -eq "$split_rows" &&
    echo "Verification passed!"


Original file and split file counts:
   20001 bank-churn.csv
    6668 part1.csv
    6668 part2.csv
    6667 part3.csv
   40004 total
Original customer records: 20000
Customer records after splitting: 20000
Verification passed!


## Exercise 3: Select rows by what they contain, and count them

`grep <pattern> <file>` prints the lines that match; `grep -v` prints the ones that don't;
`grep -c` counts instead of printing.

1. Write a CSV holding only the German customers, header line included.
2. What fraction of all customers are German?
3. Of the German customers, what fraction left the bank? The last column, `Exited`, is 1 for
   customers who left. A row for a German customer who left ends in `,1`.

Hints: pipe `grep` into `grep` to require two things at once, and into `wc -l` to count.
Watch out for `Germany` appearing in a surname — check whether that can happen here, and say
how you checked.

In [4]:

%%bash

# Exercise 3
# I used grep to find customers from Germany and saved the matching rows in a separate CSV file.
# Then I counted the German customers and compared them in order to the total number of customers in the dataset.
# I also counted how many German customers left the bank and used awk to calculate the percentages.

cd commandline_work

# Create the CSV with its original header
head -n 1 bank-churn.csv > german-customers.csv

# Find Germany in the fifth column (Geography)
tail -n +2 bank-churn.csv |
    grep -E '^([^,]*,){4}Germany,' >> german-customers.csv

# Count all customers, excluding the header
total=$(tail -n +2 bank-churn.csv | wc -l | tr -d ' ')

# Count German customers
german=$(tail -n +2 german-customers.csv | wc -l | tr -d ' ')

# Count German customers who left the bank
left=$(tail -n +2 german-customers.csv |
    grep -c ',1$')

# Calculate and display percentages
echo "Total customers: $total"
echo "German customers: $german"
echo "German customers who left: $left"

awk -v g="$german" -v t="$total" \
    'BEGIN {printf "Percentage German: %.2f%%\n", 100*g/t}'

awk -v l="$left" -v g="$german" \
    'BEGIN {printf "German customers who left: %.2f%%\n", 100*l/g}'

# Check if Germany appears in the surname column
echo "Surnames containing Germany:"
tail -n +2 bank-churn.csv |
    cut -d ',' -f 3 |
    grep -ic 'Germany' || true

# Verify the file was created
echo "German CSV row count (including header):"
wc -l german-customers.csv


Total customers: 20000
German customers: 4155
German customers who left: 1561
Percentage German: 20.77%
German customers who left: 37.57%
Surnames containing Germany:
0
German CSV row count (including header):
    4156 german-customers.csv


## Exercise 4: Counting lines is not counting matches

In the grades file, some exam scores are written `68-sent`, the grader's note that the score
had been emailed to the student. Count them.

Careful: `wc -l` counts *lines*, and a line can hold more than one `-sent`. Ask `grep` for
the number of matches rather than the number of matching lines — read `man grep` for `-o`
and for `-c`, and work out which one you need here. Compare the two answers on this file. They agree, which tells you something about the file: no line carries two of them. Describe a row that would make them disagree.

In [5]:

%%bash

# Exercise 4
# I used grep to find scores containing "-sent" and The grep -c command counts lines with a match.
# Used the grep -o command which counts each occurrence separately.
# I compared both results to see whether any line contains more than one occurrence.

cd commandline_work

echo "Lines containing -sent:"
grep -c -- '-sent' Data-1401-Grades.csv

echo "Total occurrences of -sent:"
grep -o -- '-sent' Data-1401-Grades.csv | wc -l

echo "Example rows containing -sent:"
grep -- '-sent' Data-1401-Grades.csv | head -5


Lines containing -sent:
12
Total occurrences of -sent:
      12
Example rows containing -sent:
100,96,46,,68-sent,,,,
100,100,90,,88-sent,,,,
93,100,24,,96-sent,,,,
0,81,0,0,43-sent,,,,
95,100,105,,96-sent,,,,


## Exercise 5: Clean a file with `sed`

`sed` replaces text as it passes through. Work on the grades file:

1. Its first line is a row of notes about how each lab was graded, not the header. Write a
   new file with everything **except** that first line.
2. In that new file, make these replacements, and write the result to another CSV:
   * `no solution` becomes empty
   * `did not submit the Lab` becomes empty
   * `did not submit the exam` becomes empty
   * `68-sent` becomes `68`, and likewise for every other score written that way

Hints:

* For part 1, `tail` counts from the start of the file if you ask it the right way:
  `man tail`, look for `-n +K`.
* The form is `sed 's/old/new/g' file`; `g` means every match on the line, not just the first.
* Chain several `sed` commands with pipes.
* The last replacement isn't a fixed string, since the number changes from row to row. You
  want to remove the `-sent` and keep the number, so match only `-sent`.

Compare your part 1 output against `Lectures/Lecture.11/Data-1401-Grades-Fixed.csv`, the
file Lecture 11 hands you: `cmp` or `diff` will tell you whether they're identical. You have
just made it yourself.

This is the same cleaning Lecture 11 does in pandas. Neither way is the right one. The shell
is quicker for a look and a one-off fix; pandas is where you go when the cleaning becomes
part of an analysis you will run again.

In [6]:

%%bash

# Exercise 5
# I used tail to remove the first line of the grades file because it contains notes instead of column names.
# I then used sed to replace missing-value descriptions with empty values and remove "-sent" from the grades.
# Then I saved the results in new CSV files to keep the original.
# Lastly I also checked whether the cleaning worked.

cd commandline_work

# Remove the first line
tail -n +2 Data-1401-Grades.csv > grades-no-notes.csv

echo "Checking the new header:"
head -n 2 grades-no-notes.csv

# Compare with the professor's reference file
echo "Comparing with lecture reference:"
if cmp -s grades-no-notes.csv \
    ../../../Lectures/Lecture.11/Data-1401-Grades-Fixed.csv
then
    echo "Files match!"
else
    echo "Files differ. Check the original and reference files."
fi

#  Clean the missing values and grade labels
sed \
    -e 's/no solution//g' \
    -e 's/did not submit the Lab//g' \
    -e 's/did not submit the exam//g' \
    -e 's/-sent//g' \
    grades-no-notes.csv > grades-cleaned.csv

# Check for the original unwanted text
echo "Checking for remaining unwanted text:"

if grep -E 'no solution|did not submit the Lab|did not submit the exam|-sent' grades-cleaned.csv
then
    echo "Some unwanted text remains."
else
    echo "Cleaning verification passed!"
fi

# Display a few rows of the cleaned dataset
echo "First five rows of cleaned data:"
head -n 5 grades-cleaned.csv


Checking the new header:
Lab 1,Lab 2,Lab 3,Lab 4,Exam 1,Lab 5,,Lab 7,Exam 2
,0,0,,0,0,0,0,0
Comparing with lecture reference:
Files match!
Checking for remaining unwanted text:
Cleaning verification passed!
First five rows of cleaned data:
Lab 1,Lab 2,Lab 3,Lab 4,Exam 1,Lab 5,,Lab 7,Exam 2
,0,0,,0,0,0,0,0
,,-,,,,,,
100,96,46,,68,,,,
Completed the lab,"ex4- attempted,wrong solution -2


## Exercise 6: Drop the columns that shouldn't be there

`bank-churn.csv` starts with `id`, `CustomerId` and `Surname`. None of them tells you
anything about whether a customer leaves — an id is just bookkeeping, and if a model appears
to learn something from one, it has learned about how the file was built, not about
customers. Dropping them is a real step in preparing data.

Write a new CSV without those three columns.

Hints: `cut -f` selects fields by number and `-d ','` sets the separator, which a CSV needs.
`cut` takes ranges: `-f 4-` means "from the fourth field on".

In [11]:


%%bash

# Exercise 6
# I used cut to remove the first three columns,because they contain customer identification information.
# I kept all the columns starting from column four and saved the cleaned data in a new CSV file.
# I also checked that the number of rows stayed the same.

cd commandline_work

# Remove id, CustomerId, and Surname
cut -d ',' -f 4- bank-churn.csv > bank-churn-cleaned.csv

echo "Original columns:"
head -n 1 bank-churn.csv

echo "Columns after removing identifiers:"
head -n 1 bank-churn-cleaned.csv

echo "Row counts:"
wc -l bank-churn.csv bank-churn-cleaned.csv

echo "First three customer records:"
head -n 4 bank-churn-cleaned.csv



Original columns:
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
Columns after removing identifiers:
CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
Row counts:
   20001 bank-churn.csv
   20001 bank-churn-cleaned.csv
   40002 total
First three customer records:
CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
668,France,Male,33.0,3,0.0,2,1.0,0.0,181449.97,0
627,France,Male,33.0,1,0.0,2,1.0,1.0,49503.5,0
678,France,Male,40.0,10,0.0,2,1.0,0.0,184866.69,0


## Exercise 7: Arithmetic on a column

Use `heart.csv`, which is all numbers.

1. Compute each patient's `trestbps + chol + thalach` (resting blood pressure, cholesterol,
   maximum heart rate) and write the totals to a file.
2. How many patients have `target` equal to 1?

Hints:

* `cut` gets you just the columns you want. Use the numbered header from Exercise 1.
* `tr ',' '+'` turns `145,233,150` into `145+233+150`.
* `bc` is a calculator that reads expressions and prints answers. Pipe into it.
* The header line is text, not numbers, so drop it first, the way you did in Exercise 5.
* `heart.csv` begins with an invisible marker (a byte-order mark) before `age`. It does no
  harm here, but if a command behaves oddly on the first column, that is why.

In [8]:

%%bash

# Exercise 7
# I used cut to select three numerical columns from the heart disease dataset.
# Then used tr to replace commas with addition signs.And also used bc to calculate the total for each patient.
# I saved these totals in a separate text file. And counted how many patients had a target of 1.

cd commandline_work

# Calculate the total for each patient
tail -n +2 heart.csv |
    cut -d ',' -f 4,5,8 |
    tr ',' '+' |
    tr -d '\r' |
    bc > heart-totals.txt

# Display the first five totals
echo "First five patient totals:"
head -n 5 heart-totals.txt

# Check the total number of calculations
echo "Number of patient totals:"
wc -l heart-totals.txt

# Count patients with target = 1
echo "Patients with target = 1:"
tail -n +2 heart.csv |
    cut -d ',' -f 14 |
    tr -d '\r' |
    grep -c '^1$'

# Count the total number of patients
echo "Total patients:"
tail -n +2 heart.csv | wc -l


First five patient totals:
528
567
506
534
637
Number of patient totals:
     303 heart-totals.txt
Patients with target = 1:
165
Total patients:
     303


## Exercise 8: Sort

1. Sort `bank-churn.csv` by `Balance`, largest first, keeping the header at the top, and
   write the result to a new CSV.
2. Who are the 5 customers with the largest balances, and what countries are they in?

Hints: `sort` with `-n` (numeric), `-r` (reverse), `-t ','` (the separator) and `-k` (which
field). `man sort` is worth reading for `-k`, the fiddly one. Keep the header out of the
sort the same way as before, then put it back with `>` and `>>`.

In [9]:

%%bash

# Exercise 8
# I used sort to organize the bank customers by balance, used -n to sort numerically and -r for descending order.
# I used -t to specify commas and -k to select column 9, then kept the header separate so it would stay at the top.
# After that I used head and cut to find the five customers with the highest balances and their countries.

cd commandline_work

# Save the header first
head -n 1 bank-churn.csv > bank-churn-sorted.csv

# Sort customer records by Balance (column 9)
tail -n +2 bank-churn.csv |
    sort -t ',' -k9,9nr >> bank-churn-sorted.csv

# Display the header and first five sorted records
echo "Top five records by balance:"
head -n 6 bank-churn-sorted.csv

# Show customer name, country, and balance
echo ""
echo "Top five customers (Surname, Geography, Balance):"
head -n 6 bank-churn-sorted.csv |
    cut -d ',' -f 3,5,9

# Verify the original and sorted files have the same row count
echo ""
echo "Original and sorted row counts:"
wc -l bank-churn.csv bank-churn-sorted.csv


Top five records by balance:
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
12268,15757408,Ibezimako,655,Spain,Male,38.0,3,250898.09,3,1.0,1.0,81054.0,0
8949,15587596,Nebeuwa,572,Spain,Male,31.0,5,238387.56,3,1.0,0.0,21646.2,1
3968,15768451,Nott,681,France,Female,40.0,8,222267.63,1,1.0,0.0,186281.0,1
16904,15620268,Thomson,495,Germany,Male,43.0,3,212696.32,1,1.0,0.0,115268.86,0
5031,15690540,Udinesi,532,France,Male,40.0,9,212314.03,1,1.0,1.0,148814.54,0

Top five customers (Surname, Geography, Balance):
Surname,Geography,Balance
Ibezimako,Spain,250898.09
Nebeuwa,Spain,238387.56
Nott,France,222267.63
Thomson,Germany,212696.32
Udinesi,France,212314.03

Original and sorted row counts:
   20001 bank-churn.csv
   20001 bank-churn-sorted.csv
   40002 total


## Exercise 9: Where the command line gives up

Open `2012_Workplace_Fatalities_by_State.csv` in a text editor and look at the first few
lines. Then, at the prompt:

1. How many lines does `wc -l` report?
2. How many US states are in the file?
3. Print the first column with `cut -d ',' -f 1`. What do you get?

The two counts in 1 and 2 disagree, and the `cut` output is wrong. Work out why, and write
a sentence or two explaining it. Two things in this file break the assumption every tool you
used today makes, that a line is a row and a comma separates fields.

This is the real lesson of the lab. These tools are fast and they are always there, but they
treat a CSV as lines of text. When the file's own format fights back, stop, and use something
that understands CSV — which is where `pandas.read_csv` comes in, in Lecture 11 and Lab 7.

In [14]:

%%bash

# Exercise 9
# I examined the workplace fatalities dataset in order to understand why basic CSV commands can give wrong results.
# I used wc to count physical lines in the file and grep to identify the 50 state records.
# Then cut to inspect the first column. I found that quoted fields contain line breaks and commas,
#which cause basic text commands to split the data incorrectly.

cd commandline_work

echo "Number of physical lines:"
wc -l 2012_Workplace_Fatalities_by_State.csv

echo ""
echo "Number of state records:"
grep -Ec '^"[A-Za-z ]+$' \
    2012_Workplace_Fatalities_by_State.csv

echo ""
echo "First 12 lines of the file:"
head -n 12 2012_Workplace_Fatalities_by_State.csv

echo ""
echo "First column using cut:"
cut -d ',' -f 1 \
    2012_Workplace_Fatalities_by_State.csv |
    head -n 12

echo ""
echo "Last 6 physical lines:"
tail -n 6 2012_Workplace_Fatalities_by_State.csv


Number of physical lines:
     105 2012_Workplace_Fatalities_by_State.csv

Number of state records:
50

First 12 lines of the file:
State,"Number of Fatalities, 2012","Rate of Fatalities, 2012","State Rank, Fatalities 2012",Number of Injuries/Illnesses 2012,Injuries/Illnesses 2012 Rate,Penalties FY 2013 (Average $),Penalties FY 2013 (Rank),Inspectors,Years to Inspect Each Workplace Once,State or Federal Program
"South Carolina
(33.99882060100049, -81.04536765699964)",63,3.5,25,36200,3,492,49,24,111,State
"West Virginia
(38.665511497000466, -80.71263935099967)",49,6.9,46,19800,4.1,1798,27,7,173,Federal
"Massachusetts
(42.27687306500047, -72.08268985899963)",44,1.4,1,69700,3.1,1929,21,33,123,Federal
"Tennessee
(35.680943063000484, -85.77448642199965)",101,3.8,30,65100,3.5,727,45,30,82,State
"Oklahoma
(35.472034350000456, -97.52106845499969)",97,6.1,42,39000,3.6,1872,24,19,131,Federal
"Illinois

First column using cut:
State
"South Carolina
(33.99882060100049
"West Virginia
(38.6655114970

## Submitting

Same as every lab: copy this notebook to a new name before working in it, and push the copy
to your fork.

```bash
cp Labs/Lab.5/Lab.5.CommandLine.ipynb Labs/Lab.5/Lab.5.CommandLine.solution.ipynb
```

Each exercise's cell should hold the command or commands you used, and where the exercise
asks a question — a fraction, a count, an explanation — write the answer next to them.

**Do not commit the CSV files you create.** They are working files; your commands are the
submission.
